In [21]:
!pip install gradio
!pip install langchain
!pip install langchain-classic
!pip install langchain-openai
!pip install langchain-core
!pip install langchain-community
!pip install langchain-chroma
!pip install langchain-text-splitters
!pip install pypdf

In [22]:
import os
import gradio as gr
import random
import time
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_community.chat_message_histories.in_memory import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [23]:
# Setup
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [24]:
# Initialize the language model and embeddings
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    api_key=OPENAI_API_KEY
)

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

In [25]:
# Read/Load policy in vector database
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

document = PyPDFLoader("1728286846_the_nestle_hr_policy_pdf_2012.pdf").load()
chunks = text_splitter.split_documents(document)
vector_store = Chroma.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever()

In [26]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system", 
            """
            You are an HR assistant for answering questions and queries about Nestlé's HR reports efficiently. 
            Use the provided context to respond. 
            If the answer isn't clear, acknowledge that you don't know. 
            Limit your response to three concise sentences. 
            {context}
        
            """
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ]
)


In [27]:
qa_chain = create_stuff_documents_chain(llm, prompt_template)
rag_chain = create_retrieval_chain(retriever, qa_chain)


In [28]:
# Chat History
history_for_chain = ChatMessageHistory()

chain_with_history = RunnableWithMessageHistory(
    rag_chain,
    lambda session_id: history_for_chain,
    input_messages_key="input",
    history_messages_key="chat_history"
)

In [30]:
def chat(question):
    # response = rag_chain.invoke({"input":question})
    response = (
        chain_with_history.invoke(
            {"input": question},
            {
                "configurable":{"session_id":"abc123"}
            }
        )
    )
    return response["answer"]

demo = gr.Interface(
    fn=chat,
    inputs=["text"],
    outputs=["text"]
)

with gr.Blocks() as demo:
    chatbot = gr.Chatbot()
    msg = gr.Textbox()
    clear = gr.ClearButton([msg, chatbot])

    def respond(message, chat_history):
        response = chat(message)
        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": response})
        time.sleep(2)
        return "", chat_history
    
    msg.submit(respond, [msg, chatbot], [msg, chatbot])


In [31]:
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://98740f2a0e2865398f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Error in RootListenersTracer.on_chain_end callback: KeyError('output')
Error in RootListenersTracer.on_chain_end callback: KeyError('output')
Error in RootListenersTracer.on_chain_end callback: KeyError('output')
